# AKANE E11-1：物の語そのものの「普通の意味」を、Gemmaに選ばせる

このノートは **材料を取るだけ** です（比べは結果のファイルを読んで別に行います）。計画：plans/E11-1_usual_sense.md。

- ① WordNet の名詞 2,000語で、文なしで「一番よく使う意味」を選ばせ、SemCor の回数（人が意味を付けた文で数えた回数）と照らします。
- ② E8n の物の語（約900語）で、同じ聞き方で意味を選ばせます。NLI と鎖は E8n の結果をそのまま使うので、流し直しません。
- ②の正解（常識の判定）はこのノートに入っていません。①の正解（SemCor の回数）はノートの中で数えますが、Gemma には見せません（候補はくじで並べ替えて見せます）。
- 手で書いた語の一覧はありません。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**。
2. 鍵マークの `HF_TOKEN` にアクセスを許可（E6〜E10と同じ）。
3. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
4. 終わると、ドライブの `AKANE_アーキテクチャ/E11-1_results/E11-1_results.json` ができます。
5. 時間の見込み：20〜35分。途中で切れても、もう一度「すべてのセルを実行」すれば済んだ所は飛ばします。

In [ ]:
import os, json, time, subprocess, sys
import numpy as np
from google.colab import drive, userdata
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/AKANE_アーキテクチャ"
OUT = f"{BASE}/E11-1_results"
os.makedirs(OUT, exist_ok=True)
E8N = f"{BASE}/E8n_results/E8n_results.json"
assert os.path.exists(E8N), "見つかりません: " + E8N
GEMMA_ID = "google/gemma-2-2b-it"   # E6〜E10と同じ
EMB_ID = "sentence-transformers/all-mpnet-base-v2"   # E8と同じ
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "nltk"], check=False)

In [ ]:
# 中身（analysis/e11/e111_core.py と同じ。手書きの語の一覧はありません）
"""E11-1 の中身（計画 plans/E11-1_usual_sense.md）。手書きの語の一覧はない。
- semcor_items：WordNet の名詞から、決まりで問題の語を選ぶ（正解は SemCor の回数が一番多い意味）。
- jobs_usual / jobs_leak：Gemma への聞き方（くじの順と逆順の2通り）。
- merge：2通りの答えを元の順に戻して平均する。"""
import numpy as np

MAX_SENSES = 8   # E8 と同じ（候補の番号が1桁で済む）


def semcor_items(wn, n=2000, seed=0, min_top=3):
    """意味が2〜MAX_SENSES個の名詞で、一番多い意味の回数が min_top 以上、かつ2番目より多い物から n 語をくじで選ぶ。
    defs・counts は WordNet の元の順（＝SemCor の回数順に近い）。見せる時はくじで並べ替える。"""
    pool = []
    for w in sorted({l for l in wn.all_lemma_names(pos="n")}):
        if not (len(w) >= 3 and w.isascii() and w.isalpha() and w.islower()):
            continue   # 語の形の決まり（複合語・固有名・数字・1〜2文字の略語や文字名を外す）
        ss = wn.synsets(w, pos="n")
        if not 2 <= len(ss) <= MAX_SENSES:
            continue
        cnt = []
        for s in ss:
            c = [l.count() for l in s.lemmas() if l.name().lower() == w]
            cnt.append(int(sum(c)))
        top = sorted(cnt, reverse=True)
        if top[0] < min_top or top[0] == top[1]:
            continue
        pool.append({"w": w, "syn": [s.name() for s in ss], "defs": [s.definition() for s in ss], "counts": cnt,
                     "gold": int(np.argmax(cnt))})
    rng = np.random.default_rng(seed)
    pick = sorted(rng.choice(len(pool), size=min(n, len(pool)), replace=False).tolist())
    out = [pool[i] for i in pick]
    for it in out:
        it["pool"] = len(pool)
    return out


def _opts(defs, order):
    return "\n".join(f"{j + 1}. {defs[i]}" for j, i in enumerate(order))


def prompt_usual(w, defs, order):
    return (f"Here are the meanings of the English word \"{w}\":\n{_opts(defs, order)}\n\n"
            f"Which meaning is the most common one in everyday English?\nAnswer with the number only.")


def prompt_leak(w, defs, order, c):
    W = w[:1].upper() + w[1:]
    return (f"Here are the meanings of the English word \"{w}\":\n{_opts(defs, order)}\n\n"
            f"In the sentence \"{W} is a type of {c}.\", which meaning of \"{w}\" is used?\nAnswer with the number only.")


def _orders(n, rng):
    o = rng.permutation(n).tolist()
    return [o, o[::-1]]


def jobs_usual(pairs, seed=0):
    """pairs: [(語, 説明文の一覧)]。1語につき2つ（くじの順・逆順）の [聞き方, 見せた順（元の番号）]。"""
    rng = np.random.default_rng(seed)
    out = []
    for w, defs in pairs:
        defs = defs[:MAX_SENSES]
        for o in _orders(len(defs), rng):
            out.append([prompt_usual(w, defs, o), o])
    return out


def jobs_leak(triples, seed=0):
    """triples: [(語, 説明文の一覧, 分類名)]。参考 R4 用（分類を見せる）。"""
    rng = np.random.default_rng(seed)
    out = []
    for w, defs, c in triples:
        defs = defs[:MAX_SENSES]
        for o in _orders(len(defs), rng):
            out.append([prompt_leak(w, defs, o, c), o])
    return out


def merge(jobs, probs, n):
    """jobs[2i], jobs[2i+1] が i 番目の語。見せた順の値を元の順に戻し、2通りを平均する。"""
    assert len(jobs) == len(probs) == 2 * n
    out = []
    for i in range(n):
        acc = None
        for j in (2 * i, 2 * i + 1):
            order, p = jobs[j][1], probs[j]
            v = np.zeros(len(order))
            for pos, orig in enumerate(order):
                v[orig] = p[pos]
            acc = v if acc is None else acc + v
        out.append((acc / 2).round(4).tolist())
    return out

In [ ]:
OLD = json.loads(r'''[{"id": 0, "ent": "barbell", "catw": "tool"}, {"id": 1, "ent": "tenement", "catw": "building"}, {"id": 2, "ent": "porpoise", "catw": "dog"}, {"id": 3, "ent": "juvenile", "catw": "boat"}, {"id": 4, "ent": "barbell", "catw": "mammal"}, {"id": 5, "ent": "grocer", "catw": "living thing"}, {"id": 6, "ent": "gramophone", "catw": "game"}, {"id": 7, "ent": "hooch", "catw": "artist"}, {"id": 8, "ent": "bilingual", "catw": "person"}, {"id": 9, "ent": "minnow", "catw": "disease"}, {"id": 10, "ent": "gazelle", "catw": "food"}, {"id": 11, "ent": "gasket", "catw": "game"}, {"id": 12, "ent": "isolationist", "catw": "living thing"}, {"id": 13, "ent": "clamp", "catw": "tool"}, {"id": 14, "ent": "copycat", "catw": "body part"}, {"id": 15, "ent": "peshmerga", "catw": "living thing"}, {"id": 16, "ent": "tabor", "catw": "boat"}, {"id": 17, "ent": "stepmother", "catw": "person"}, {"id": 18, "ent": "terrier", "catw": "bird"}, {"id": 19, "ent": "preteen", "catw": "metal"}, {"id": 20, "ent": "composer", "catw": "person"}, {"id": 21, "ent": "suitor", "catw": "living thing"}, {"id": 22, "ent": "leaker", "catw": "person"}, {"id": 23, "ent": "actuator", "catw": "machine"}, {"id": 24, "ent": "lally", "catw": "tool"}, {"id": 25, "ent": "epileptic", "catw": "disease"}, {"id": 26, "ent": "pew", "catw": "man-made object"}, {"id": 27, "ent": "microphone", "catw": "equipment"}, {"id": 28, "ent": "felt", "catw": "material"}, {"id": 29, "ent": "primrose", "catw": "living thing"}, {"id": 30, "ent": "quartermaster", "catw": "person"}, {"id": 31, "ent": "delegate", "catw": "person"}, {"id": 32, "ent": "grubby", "catw": "toy"}, {"id": 33, "ent": "prince", "catw": "artist"}, {"id": 34, "ent": "locomotive", "catw": "man-made object"}, {"id": 35, "ent": "lobbyist", "catw": "living thing"}, {"id": 36, "ent": "workman", "catw": "footwear"}, {"id": 37, "ent": "stupid", "catw": "person"}, {"id": 38, "ent": "watchman", "catw": "body part"}, {"id": 39, "ent": "headliner", "catw": "equipment"}, {"id": 40, "ent": "geranium", "catw": "living thing"}, {"id": 41, "ent": "guerrilla", "catw": "weapon"}, {"id": 42, "ent": "sheeting", "catw": "food"}, {"id": 43, "ent": "consulate", "catw": "device"}, {"id": 44, "ent": "dinky", "catw": "artist"}, {"id": 45, "ent": "caterer", "catw": "sport"}, {"id": 46, "ent": "rickey", "catw": "tree"}, {"id": 47, "ent": "headmistress", "catw": "worker"}, {"id": 48, "ent": "educator", "catw": "worker"}, {"id": 49, "ent": "golf", "catw": "metal"}, {"id": 50, "ent": "velvet", "catw": "man-made object"}, {"id": 51, "ent": "biographer", "catw": "disease"}, {"id": 52, "ent": "zoologist", "catw": "living thing"}, {"id": 53, "ent": "doge", "catw": "living thing"}, {"id": 54, "ent": "mimic", "catw": "reptile"}, {"id": 55, "ent": "passerby", "catw": "organ"}, {"id": 56, "ent": "dieter", "catw": "fish"}, {"id": 57, "ent": "niqab", "catw": "reptile"}, {"id": 58, "ent": "bulldozer", "catw": "machine"}, {"id": 59, "ent": "deflector", "catw": "equipment"}, {"id": 60, "ent": "schoolgirl", "catw": "metal"}, {"id": 61, "ent": "falconer", "catw": "sport"}, {"id": 62, "ent": "marlin", "catw": "drink"}, {"id": 63, "ent": "plumber", "catw": "bird"}, {"id": 64, "ent": "sarcophagus", "catw": "man-made object"}, {"id": 65, "ent": "amputee", "catw": "person"}, {"id": 66, "ent": "kimono", "catw": "fish"}, {"id": 67, "ent": "temp", "catw": "worker"}, {"id": 68, "ent": "propeller", "catw": "reptile"}, {"id": 69, "ent": "wombat", "catw": "building"}, {"id": 70, "ent": "paintbrush", "catw": "tool"}, {"id": 71, "ent": "womanizer", "catw": "body part"}, {"id": 72, "ent": "greenhouse", "catw": "man-made object"}, {"id": 73, "ent": "tactician", "catw": "food"}, {"id": 74, "ent": "achiever", "catw": "fruit"}, {"id": 75, "ent": "moll", "catw": "person"}, {"id": 76, "ent": "bonsai", "catw": "plant"}, {"id": 77, "ent": "underdog", "catw": "tree"}, {"id": 78, "ent": "loom", "catw": "car"}, {"id": 79, "ent": "individualist", "catw": "person"}, {"id": 80, "ent": "easel", "catw": "sport"}, {"id": 81, "ent": "finalist", "catw": "athlete"}, {"id": 82, "ent": "sagebrush", "catw": "weapon"}, {"id": 83, "ent": "uniform", "catw": "equipment"}, {"id": 84, "ent": "bonsai", "catw": "animal"}, {"id": 85, "ent": "receptionist", "catw": "worker"}, {"id": 86, "ent": "commoner", "catw": "worker"}, {"id": 87, "ent": "modernist", "catw": "person"}, {"id": 88, "ent": "cuppa", "catw": "container"}, {"id": 89, "ent": "cornerback", "catw": "athlete"}, {"id": 90, "ent": "minnow", "catw": "game"}, {"id": 91, "ent": "neurologist", "catw": "worker"}, {"id": 92, "ent": "kitten", "catw": "mammal"}, {"id": 93, "ent": "undercarriage", "catw": "device"}, {"id": 94, "ent": "reporter", "catw": "worker"}, {"id": 95, "ent": "critter", "catw": "equipment"}, {"id": 96, "ent": "granddaughter", "catw": "mammal"}, {"id": 97, "ent": "crusher", "catw": "machine"}, {"id": 98, "ent": "accomplice", "catw": "athlete"}, {"id": 99, "ent": "dubbing", "catw": "device"}, {"id": 100, "ent": "tortoise", "catw": "reptile"}, {"id": 101, "ent": "jukebox", "catw": "equipment"}, {"id": 102, "ent": "restaurant", "catw": "building"}, {"id": 103, "ent": "sectional", "catw": "man-made object"}, {"id": 104, "ent": "heifer", "catw": "living thing"}, {"id": 105, "ent": "kea", "catw": "living thing"}, {"id": 106, "ent": "minion", "catw": "living thing"}, {"id": 107, "ent": "opportunist", "catw": "person"}, {"id": 108, "ent": "conformist", "catw": "artist"}, {"id": 109, "ent": "burglar", "catw": "body part"}, {"id": 110, "ent": "buddy", "catw": "man-made object"}, {"id": 111, "ent": "wraparound", "catw": "metal"}, {"id": 112, "ent": "vise", "catw": "tool"}, {"id": 113, "ent": "helicopter", "catw": "man-made object"}, {"id": 114, "ent": "calender", "catw": "tool"}, {"id": 115, "ent": "snob", "catw": "game"}, {"id": 116, "ent": "drunkard", "catw": "living thing"}, {"id": 117, "ent": "bandicoot", "catw": "disease"}, {"id": 118, "ent": "musket", "catw": "vehicle"}, {"id": 119, "ent": "grader", "catw": "sport"}, {"id": 120, "ent": "briefcase", "catw": "vegetable"}, {"id": 121, "ent": "alternator", "catw": "vehicle"}, {"id": 122, "ent": "wallaby", "catw": "plant"}, {"id": 123, "ent": "interceptor", "catw": "aircraft"}, {"id": 124, "ent": "carbine", "catw": "equipment"}, {"id": 125, "ent": "spitfire", "catw": "living thing"}, {"id": 126, "ent": "huntress", "catw": "worker"}, {"id": 127, "ent": "heathen", "catw": "flower"}, {"id": 128, "ent": "senna", "catw": "man-made object"}, {"id": 129, "ent": "macho", "catw": "animal"}, {"id": 130, "ent": "beloved", "catw": "insect"}, {"id": 131, "ent": "wallaby", "catw": "living thing"}, {"id": 132, "ent": "accountant", "catw": "game"}, {"id": 133, "ent": "shogun", "catw": "fabric"}, {"id": 134, "ent": "parsonage", "catw": "boat"}, {"id": 135, "ent": "timepiece", "catw": "tool"}, {"id": 136, "ent": "biter", "catw": "living thing"}, {"id": 137, "ent": "rugby", "catw": "equipment"}, {"id": 138, "ent": "bedroom", "catw": "building"}, {"id": 139, "ent": "wombat", "catw": "tree"}, {"id": 140, "ent": "everlasting", "catw": "game"}, {"id": 141, "ent": "auctioneer", "catw": "material"}, {"id": 142, "ent": "theorist", "catw": "person"}, {"id": 143, "ent": "skater", "catw": "person"}, {"id": 144, "ent": "mariner", "catw": "metal"}, {"id": 145, "ent": "freelancer", "catw": "fish"}, {"id": 146, "ent": "dubbing", "catw": "game"}, {"id": 147, "ent": "simulator", "catw": "game"}, {"id": 148, "ent": "circuitry", "catw": "sport"}, {"id": 149, "ent": "galleon", "catw": "boat"}, {"id": 150, "ent": "steed", "catw": "animal"}, {"id": 151, "ent": "mosquito", "catw": "weapon"}, {"id": 152, "ent": "legate", "catw": "game"}, {"id": 153, "ent": "maxi", "catw": "room"}, {"id": 154, "ent": "bodybuilder", "catw": "animal"}, {"id": 155, "ent": "newlywed", "catw": "living thing"}, {"id": 156, "ent": "finch", "catw": "material"}, {"id": 157, "ent": "flea", "catw": "vegetable"}, {"id": 158, "ent": "betel", "catw": "tree"}, {"id": 159, "ent": "microphone", "catw": "sport"}, {"id": 160, "ent": "airliner", "catw": "boat"}, {"id": 161, "ent": "commodore", "catw": "machine"}, {"id": 162, "ent": "billionaire", "catw": "body part"}, {"id": 163, "ent": "roadhouse", "catw": "game"}, {"id": 164, "ent": "hater", "catw": "person"}, {"id": 165, "ent": "marc", "catw": "fruit"}, {"id": 166, "ent": "tugboat", "catw": "disease"}, {"id": 167, "ent": "blouse", "catw": "fish"}, {"id": 168, "ent": "houseboat", "catw": "device"}, {"id": 169, "ent": "bovine", "catw": "toy"}, {"id": 170, "ent": "porpoise", "catw": "athlete"}, {"id": 171, "ent": "everlasting", "catw": "plant"}, {"id": 172, "ent": "tomahawk", "catw": "man-made object"}, {"id": 173, "ent": "national", "catw": "boat"}, {"id": 174, "ent": "underpants", "catw": "man-made object"}, {"id": 175, "ent": "hitchhiker", "catw": "worker"}, {"id": 176, "ent": "cutthroat", "catw": "plant"}, {"id": 177, "ent": "theologian", "catw": "reptile"}, {"id": 178, "ent": "facilitator", "catw": "vegetable"}, {"id": 179, "ent": "rickey", "catw": "man-made object"}, {"id": 180, "ent": "songwriter", "catw": "dish"}, {"id": 181, "ent": "extremist", "catw": "fruit"}, {"id": 182, "ent": "transvestite", "catw": "sport"}, {"id": 183, "ent": "warship", "catw": "fruit"}, {"id": 184, "ent": "spokeswoman", "catw": "body part"}, {"id": 185, "ent": "brill", "catw": "animal"}, {"id": 186, "ent": "secessionist", "catw": "metal"}, {"id": 187, "ent": "beloved", "catw": "fabric"}, {"id": 188, "ent": "impala", "catw": "sport"}, {"id": 189, "ent": "firearm", "catw": "man-made object"}, {"id": 190, "ent": "mathematician", "catw": "worker"}, {"id": 191, "ent": "buttercup", "catw": "man-made object"}, {"id": 192, "ent": "notary", "catw": "game"}, {"id": 193, "ent": "missus", "catw": "boat"}, {"id": 194, "ent": "freelancer", "catw": "living thing"}, {"id": 195, "ent": "beret", "catw": "toy"}, {"id": 196, "ent": "georgette", "catw": "clothing"}, {"id": 197, "ent": "parsonage", "catw": "man-made object"}, {"id": 198, "ent": "dongle", "catw": "metal"}, {"id": 199, "ent": "spacecraft", "catw": "vehicle"}, {"id": 200, "ent": "eunuch", "catw": "living thing"}, {"id": 201, "ent": "apparel", "catw": "clothing"}, {"id": 202, "ent": "messenger", "catw": "person"}, {"id": 203, "ent": "strainer", "catw": "game"}, {"id": 204, "ent": "bookkeeper", "catw": "living thing"}, {"id": 205, "ent": "clamp", "catw": "musical instrument"}, {"id": 206, "ent": "sieve", "catw": "body part"}, {"id": 207, "ent": "madame", "catw": "insect"}, {"id": 208, "ent": "bethel", "catw": "body part"}, {"id": 209, "ent": "moray", "catw": "disease"}, {"id": 210, "ent": "lemur", "catw": "animal"}, {"id": 211, "ent": "orangutan", "catw": "fruit"}, {"id": 212, "ent": "athlete", "catw": "living thing"}, {"id": 213, "ent": "vole", "catw": "worker"}, {"id": 214, "ent": "peony", "catw": "car"}, {"id": 215, "ent": "undercarriage", "catw": "man-made object"}, {"id": 216, "ent": "blacksmith", "catw": "artist"}, {"id": 217, "ent": "erica", "catw": "game"}, {"id": 218, "ent": "clinician", "catw": "person"}, {"id": 219, "ent": "beekeeper", "catw": "person"}, {"id": 220, "ent": "purveyor", "catw": "living thing"}, {"id": 221, "ent": "carbine", "catw": "man-made object"}, {"id": 222, "ent": "ani", "catw": "animal"}, {"id": 223, "ent": "buyer", "catw": "material"}, {"id": 224, "ent": "bankrupt", "catw": "man-made object"}, {"id": 225, "ent": "landlady", "catw": "flower"}, {"id": 226, "ent": "winger", "catw": "aircraft"}, {"id": 227, "ent": "meatball", "catw": "food"}, {"id": 228, "ent": "autofocus", "catw": "man-made object"}, {"id": 229, "ent": "narrator", "catw": "clothing"}, {"id": 230, "ent": "winch", "catw": "musical instrument"}, {"id": 231, "ent": "hoist", "catw": "car"}, {"id": 232, "ent": "poodle", "catw": "disease"}, {"id": 233, "ent": "hamster", "catw": "athlete"}, {"id": 234, "ent": "finch", "catw": "bird"}, {"id": 235, "ent": "shrine", "catw": "vehicle"}, {"id": 236, "ent": "ivy", "catw": "car"}, {"id": 237, "ent": "pensioner", "catw": "person"}, {"id": 238, "ent": "flatmate", "catw": "organ"}, {"id": 239, "ent": "communicator", "catw": "living thing"}, {"id": 240, "ent": "menial", "catw": "plant"}, {"id": 241, "ent": "grandparent", "catw": "food"}, {"id": 242, "ent": "chalet", "catw": "building"}, {"id": 243, "ent": "avenger", "catw": "vegetable"}, {"id": 244, "ent": "gazebo", "catw": "fish"}, {"id": 245, "ent": "marten", "catw": "living thing"}, {"id": 246, "ent": "uterus", "catw": "container"}, {"id": 247, "ent": "mockingbird", "catw": "person"}, {"id": 248, "ent": "milkman", "catw": "living thing"}, {"id": 249, "ent": "sauna", "catw": "animal"}, {"id": 250, "ent": "aviator", "catw": "man-made object"}, {"id": 251, "ent": "freeman", "catw": "worker"}, {"id": 252, "ent": "salesman", "catw": "fish"}, {"id": 253, "ent": "synchrotron", "catw": "device"}, {"id": 254, "ent": "traveler", "catw": "reptile"}, {"id": 255, "ent": "skewer", "catw": "game"}, {"id": 256, "ent": "sleuth", "catw": "metal"}, {"id": 257, "ent": "abolitionist", "catw": "artist"}, {"id": 258, "ent": "mantel", "catw": "device"}, {"id": 259, "ent": "consulate", "catw": "building"}, {"id": 260, "ent": "minster", "catw": "boat"}, {"id": 261, "ent": "informer", "catw": "fruit"}, {"id": 262, "ent": "waitress", "catw": "food"}, {"id": 263, "ent": "loudspeaker", "catw": "man-made object"}, {"id": 264, "ent": "orator", "catw": "footwear"}, {"id": 265, "ent": "drummer", "catw": "worker"}, {"id": 266, "ent": "dingo", "catw": "animal"}, {"id": 267, "ent": "sailboat", "catw": "machine"}, {"id": 268, "ent": "idealist", "catw": "disease"}, {"id": 269, "ent": "cactus", "catw": "weapon"}, {"id": 270, "ent": "basset", "catw": "animal"}, {"id": 271, "ent": "handkerchief", "catw": "man-made object"}, {"id": 272, "ent": "plumber", "catw": "tree"}, {"id": 273, "ent": "webcam", "catw": "man-made object"}, {"id": 274, "ent": "dinghy", "catw": "boat"}, {"id": 275, "ent": "mod", "catw": "metal"}, {"id": 276, "ent": "erotic", "catw": "flower"}, {"id": 277, "ent": "songbird", "catw": "room"}, {"id": 278, "ent": "sauerkraut", "catw": "fruit"}, {"id": 279, "ent": "plenipotentiary", "catw": "equipment"}, {"id": 280, "ent": "dispensary", "catw": "man-made object"}, {"id": 281, "ent": "fetus", "catw": "man-made object"}, {"id": 282, "ent": "orchid", "catw": "artist"}, {"id": 283, "ent": "basset", "catw": "dog"}, {"id": 284, "ent": "coordinator", "catw": "person"}, {"id": 285, "ent": "taster", "catw": "worker"}, {"id": 286, "ent": "swot", "catw": "sport"}, {"id": 287, "ent": "exporter", "catw": "material"}, {"id": 288, "ent": "underwear", "catw": "clothing"}, {"id": 289, "ent": "pendulum", "catw": "device"}, {"id": 290, "ent": "stoner", "catw": "bird"}, {"id": 291, "ent": "bartender", "catw": "game"}, {"id": 292, "ent": "juggler", "catw": "living thing"}, {"id": 293, "ent": "sovereign", "catw": "game"}, {"id": 294, "ent": "stepson", "catw": "mammal"}, {"id": 295, "ent": "mongoose", "catw": "living thing"}, {"id": 296, "ent": "hater", "catw": "dog"}, {"id": 297, "ent": "cutthroat", "catw": "game"}, {"id": 298, "ent": "pedophile", "catw": "food"}, {"id": 299, "ent": "ripper", "catw": "plant"}, {"id": 300, "ent": "sultan", "catw": "organ"}, {"id": 301, "ent": "assistant", "catw": "dish"}, {"id": 302, "ent": "coauthor", "catw": "vegetable"}, {"id": 303, "ent": "ladybug", "catw": "equipment"}, {"id": 304, "ent": "dredge", "catw": "man-made object"}, {"id": 305, "ent": "clarence", "catw": "vehicle"}, {"id": 306, "ent": "eft", "catw": "aircraft"}, {"id": 307, "ent": "modernist", "catw": "building"}, {"id": 308, "ent": "satchel", "catw": "man-made object"}, {"id": 309, "ent": "invalid", "catw": "game"}, {"id": 310, "ent": "jogger", "catw": "food"}, {"id": 311, "ent": "rancher", "catw": "body part"}, {"id": 312, "ent": "clary", "catw": "body part"}, {"id": 313, "ent": "recliner", "catw": "body part"}, {"id": 314, "ent": "mercenary", "catw": "worker"}, {"id": 315, "ent": "storeroom", "catw": "vegetable"}, {"id": 316, "ent": "protege", "catw": "vegetable"}, {"id": 317, "ent": "handyman", "catw": "weapon"}, {"id": 318, "ent": "wanton", "catw": "living thing"}, {"id": 319, "ent": "bloodhound", "catw": "vegetable"}, {"id": 320, "ent": "crustacean", "catw": "body part"}, {"id": 321, "ent": "electric", "catw": "car"}, {"id": 322, "ent": "itinerant", "catw": "artist"}, {"id": 323, "ent": "outhouse", "catw": "container"}, {"id": 324, "ent": "searchlight", "catw": "metal"}, {"id": 325, "ent": "stickler", "catw": "person"}, {"id": 326, "ent": "snob", "catw": "person"}, {"id": 327, "ent": "erotic", "catw": "living thing"}, {"id": 328, "ent": "violin", "catw": "musical instrument"}, {"id": 329, "ent": "plover", "catw": "bird"}, {"id": 330, "ent": "lobbyist", "catw": "boat"}, {"id": 331, "ent": "motorcyclist", "catw": "footwear"}, {"id": 332, "ent": "harlequin", "catw": "reptile"}, {"id": 333, "ent": "cheeseburger", "catw": "man-made object"}, {"id": 334, "ent": "wearer", "catw": "mammal"}, {"id": 335, "ent": "geisha", "catw": "plant"}, {"id": 336, "ent": "poppy", "catw": "sport"}, {"id": 337, "ent": "bobbin", "catw": "fruit"}, {"id": 338, "ent": "preacher", "catw": "person"}, {"id": 339, "ent": "ladybug", "catw": "insect"}, {"id": 340, "ent": "cubby", "catw": "living thing"}, {"id": 341, "ent": "neurotic", "catw": "metal"}, {"id": 342, "ent": "applicator", "catw": "device"}, {"id": 343, "ent": "hoarder", "catw": "mammal"}, {"id": 344, "ent": "escapist", "catw": "game"}, {"id": 345, "ent": "bereaved", "catw": "living thing"}, {"id": 346, "ent": "termite", "catw": "mammal"}, {"id": 347, "ent": "dragonfly", "catw": "fish"}, {"id": 348, "ent": "razor", "catw": "weapon"}, {"id": 349, "ent": "mensch", "catw": "person"}, {"id": 350, "ent": "kingfisher", "catw": "insect"}, {"id": 351, "ent": "snob", "catw": "vegetable"}, {"id": 352, "ent": "bock", "catw": "vegetable"}, {"id": 353, "ent": "accelerometer", "catw": "sport"}, {"id": 354, "ent": "electrode", "catw": "equipment"}, {"id": 355, "ent": "subcontractor", "catw": "food"}, {"id": 356, "ent": "vase", "catw": "container"}, {"id": 357, "ent": "parachute", "catw": "equipment"}, {"id": 358, "ent": "soloist", "catw": "metal"}, {"id": 359, "ent": "prankster", "catw": "person"}, {"id": 360, "ent": "missus", "catw": "person"}, {"id": 361, "ent": "gynecologist", "catw": "living thing"}, {"id": 362, "ent": "landowner", "catw": "person"}, {"id": 363, "ent": "mosquito", "catw": "mammal"}, {"id": 364, "ent": "decadent", "catw": "dish"}, {"id": 365, "ent": "toolbox", "catw": "tool"}, {"id": 366, "ent": "brandy", "catw": "food"}, {"id": 367, "ent": "housekeeper", "catw": "flower"}, {"id": 368, "ent": "convener", "catw": "material"}, {"id": 369, "ent": "moa", "catw": "animal"}, {"id": 370, "ent": "advertiser", "catw": "metal"}, {"id": 371, "ent": "spaniel", "catw": "dog"}, {"id": 372, "ent": "denture", "catw": "man-made object"}, {"id": 373, "ent": "pigeon", "catw": "person"}, {"id": 374, "ent": "chrysalis", "catw": "body part"}, {"id": 375, "ent": "vizier", "catw": "athlete"}, {"id": 376, "ent": "bandit", "catw": "device"}, {"id": 377, "ent": "dominatrix", "catw": "living thing"}, {"id": 378, "ent": "protectionist", "catw": "plant"}, {"id": 379, "ent": "cyborg", "catw": "body part"}, {"id": 380, "ent": "manger", "catw": "worker"}, {"id": 381, "ent": "stowaway", "catw": "vegetable"}, {"id": 382, "ent": "stingray", "catw": "living thing"}, {"id": 383, "ent": "immigrant", "catw": "game"}, {"id": 384, "ent": "peripheral", "catw": "animal"}, {"id": 385, "ent": "corgi", "catw": "clothing"}, {"id": 386, "ent": "trusty", "catw": "game"}, {"id": 387, "ent": "inventor", "catw": "drink"}, {"id": 388, "ent": "narc", "catw": "person"}, {"id": 389, "ent": "brazier", "catw": "drink"}, {"id": 390, "ent": "curate", "catw": "living thing"}, {"id": 391, "ent": "stagecoach", "catw": "vehicle"}, {"id": 392, "ent": "molester", "catw": "mammal"}, {"id": 393, "ent": "paramedic", "catw": "worker"}, {"id": 394, "ent": "inventor", "catw": "body part"}, {"id": 395, "ent": "synagogue", "catw": "vegetable"}, {"id": 396, "ent": "adolescent", "catw": "mammal"}, {"id": 397, "ent": "contrarian", "catw": "metal"}, {"id": 398, "ent": "swimsuit", "catw": "equipment"}, {"id": 399, "ent": "shrine", "catw": "container"}, {"id": 400, "ent": "manatee", "catw": "vehicle"}, {"id": 401, "ent": "toddler", "catw": "tool"}, {"id": 402, "ent": "introvert", "catw": "worker"}, {"id": 403, "ent": "copywriter", "catw": "clothing"}, {"id": 404, "ent": "vigilante", "catw": "person"}, {"id": 405, "ent": "cuttlefish", "catw": "animal"}, {"id": 406, "ent": "alderman", "catw": "athlete"}, {"id": 407, "ent": "hater", "catw": "boat"}, {"id": 408, "ent": "airship", "catw": "dog"}, {"id": 409, "ent": "benefactor", "catw": "person"}, {"id": 410, "ent": "financier", "catw": "worker"}, {"id": 411, "ent": "trader", "catw": "worker"}, {"id": 412, "ent": "anti", "catw": "living thing"}, {"id": 413, "ent": "sarcophagus", "catw": "tool"}, {"id": 414, "ent": "paleontologist", "catw": "worker"}, {"id": 415, "ent": "accordion", "catw": "fish"}, {"id": 416, "ent": "sphincter", "catw": "disease"}, {"id": 417, "ent": "detainee", "catw": "living thing"}, {"id": 418, "ent": "pleaser", "catw": "game"}, {"id": 419, "ent": "princess", "catw": "person"}, {"id": 420, "ent": "paramedic", "catw": "dish"}, {"id": 421, "ent": "rink", "catw": "sport"}, {"id": 422, "ent": "propagandist", "catw": "person"}, {"id": 423, "ent": "confidante", "catw": "person"}, {"id": 424, "ent": "stockbroker", "catw": "footwear"}, {"id": 425, "ent": "drosophila", "catw": "food"}, {"id": 426, "ent": "pacifist", "catw": "insect"}, {"id": 427, "ent": "suitor", "catw": "game"}, {"id": 428, "ent": "apologist", "catw": "worker"}, {"id": 429, "ent": "clubhouse", "catw": "room"}, {"id": 430, "ent": "strategist", "catw": "person"}, {"id": 431, "ent": "jeep", "catw": "container"}, {"id": 432, "ent": "survivalist", "catw": "fish"}, {"id": 433, "ent": "vermouth", "catw": "plant"}, {"id": 434, "ent": "contrarian", "catw": "animal"}, {"id": 435, "ent": "cattle", "catw": "animal"}, {"id": 436, "ent": "pensioner", "catw": "sport"}, {"id": 437, "ent": "marlin", "catw": "living thing"}, {"id": 438, "ent": "mockingbird", "catw": "bird"}, {"id": 439, "ent": "dandelion", "catw": "mammal"}, {"id": 440, "ent": "meatball", "catw": "game"}, {"id": 441, "ent": "chime", "catw": "organ"}, {"id": 442, "ent": "hornet", "catw": "worker"}, {"id": 443, "ent": "wildflower", "catw": "living thing"}, {"id": 444, "ent": "gyroscope", "catw": "machine"}, {"id": 445, "ent": "lender", "catw": "bird"}, {"id": 446, "ent": "kangaroo", "catw": "athlete"}, {"id": 447, "ent": "calico", "catw": "fabric"}, {"id": 448, "ent": "tongs", "catw": "tool"}, {"id": 449, "ent": "cadet", "catw": "dish"}, {"id": 450, "ent": "repairman", "catw": "fruit"}, {"id": 451, "ent": "clubhouse", "catw": "machine"}, {"id": 452, "ent": "hummingbird", "catw": "animal"}, {"id": 453, "ent": "bison", "catw": "mammal"}, {"id": 454, "ent": "piglet", "catw": "mammal"}, {"id": 455, "ent": "hyena", "catw": "plant"}, {"id": 456, "ent": "wife", "catw": "mammal"}, {"id": 457, "ent": "berk", "catw": "living thing"}, {"id": 458, "ent": "transistor", "catw": "fish"}, {"id": 459, "ent": "firstborn", "catw": "sport"}, {"id": 460, "ent": "interviewee", "catw": "worker"}, {"id": 461, "ent": "surrealist", "catw": "fish"}, {"id": 462, "ent": "corrective", "catw": "man-made object"}, {"id": 463, "ent": "searchlight", "catw": "equipment"}, {"id": 464, "ent": "bookkeeper", "catw": "reptile"}, {"id": 465, "ent": "takin", "catw": "fish"}, {"id": 466, "ent": "therapist", "catw": "living thing"}, {"id": 467, "ent": "communicator", "catw": "person"}, {"id": 468, "ent": "bereaved", "catw": "game"}, {"id": 469, "ent": "apparel", "catw": "weapon"}, {"id": 470, "ent": "boardroom", "catw": "room"}, {"id": 471, "ent": "goldilocks", "catw": "flower"}, {"id": 472, "ent": "manatee", "catw": "food"}, {"id": 473, "ent": "aloe", "catw": "room"}, {"id": 474, "ent": "congressman", "catw": "flower"}, {"id": 475, "ent": "ecologist", "catw": "worker"}, {"id": 476, "ent": "dahlia", "catw": "plant"}, {"id": 477, "ent": "lodger", "catw": "plant"}, {"id": 478, "ent": "flea", "catw": "animal"}, {"id": 479, "ent": "hillbilly", "catw": "living thing"}, {"id": 480, "ent": "stewardess", "catw": "equipment"}, {"id": 481, "ent": "helmsman", "catw": "athlete"}, {"id": 482, "ent": "jasmine", "catw": "vegetable"}, {"id": 483, "ent": "palmetto", "catw": "musical instrument"}, {"id": 484, "ent": "rapporteur", "catw": "organ"}, {"id": 485, "ent": "bobcat", "catw": "mammal"}, {"id": 486, "ent": "cutler", "catw": "living thing"}, {"id": 487, "ent": "hearse", "catw": "man-made object"}, {"id": 488, "ent": "spastic", "catw": "reptile"}, {"id": 489, "ent": "alcoholic", "catw": "drink"}, {"id": 490, "ent": "longhorn", "catw": "tree"}, {"id": 491, "ent": "beagle", "catw": "sport"}, {"id": 492, "ent": "impressionist", "catw": "fabric"}, {"id": 493, "ent": "inpatient", "catw": "fruit"}, {"id": 494, "ent": "emir", "catw": "boat"}, {"id": 495, "ent": "snapdragon", "catw": "living thing"}, {"id": 496, "ent": "stockholder", "catw": "furniture"}, {"id": 497, "ent": "schizophrenic", "catw": "sport"}, {"id": 498, "ent": "rapporteur", "catw": "person"}, {"id": 499, "ent": "ombudsman", "catw": "living thing"}, {"id": 500, "ent": "cleric", "catw": "plant"}, {"id": 501, "ent": "patriot", "catw": "aircraft"}, {"id": 502, "ent": "homophobe", "catw": "sport"}, {"id": 503, "ent": "rifle", "catw": "animal"}, {"id": 504, "ent": "phytoplankton", "catw": "plant"}, {"id": 505, "ent": "caveman", "catw": "person"}, {"id": 506, "ent": "exporter", "catw": "drink"}, {"id": 507, "ent": "sibling", "catw": "mammal"}, {"id": 508, "ent": "limousine", "catw": "clothing"}, {"id": 509, "ent": "lyre", "catw": "man-made object"}, {"id": 510, "ent": "audiovisual", "catw": "material"}, {"id": 511, "ent": "communicator", "catw": "sport"}, {"id": 512, "ent": "bathrobe", "catw": "metal"}, {"id": 513, "ent": "thwart", "catw": "device"}, {"id": 514, "ent": "stork", "catw": "aircraft"}, {"id": 515, "ent": "marksman", "catw": "game"}, {"id": 516, "ent": "confessional", "catw": "toy"}, {"id": 517, "ent": "monastery", "catw": "container"}, {"id": 518, "ent": "headdress", "catw": "flower"}, {"id": 519, "ent": "statesman", "catw": "living thing"}, {"id": 520, "ent": "milkman", "catw": "vegetable"}, {"id": 521, "ent": "thermometer", "catw": "tool"}, {"id": 522, "ent": "expressionist", "catw": "game"}, {"id": 523, "ent": "perpetrator", "catw": "insect"}, {"id": 524, "ent": "coupe", "catw": "container"}, {"id": 525, "ent": "councilman", "catw": "worker"}, {"id": 526, "ent": "authoritarian", "catw": "sport"}, {"id": 527, "ent": "satirist", "catw": "artist"}, {"id": 528, "ent": "antelope", "catw": "living thing"}, {"id": 529, "ent": "scraper", "catw": "device"}, {"id": 530, "ent": "ballroom", "catw": "sport"}, {"id": 531, "ent": "dunce", "catw": "flower"}, {"id": 532, "ent": "hiker", "catw": "athlete"}, {"id": 533, "ent": "playmate", "catw": "mammal"}, {"id": 534, "ent": "agitator", "catw": "vegetable"}, {"id": 535, "ent": "stiletto", "catw": "footwear"}, {"id": 536, "ent": "clock", "catw": "machine"}, {"id": 537, "ent": "bankrupt", "catw": "person"}, {"id": 538, "ent": "sophomore", "catw": "boat"}, {"id": 539, "ent": "hillbilly", "catw": "material"}, {"id": 540, "ent": "yarrow", "catw": "flower"}, {"id": 541, "ent": "caveman", "catw": "living thing"}, {"id": 542, "ent": "battleship", "catw": "person"}, {"id": 543, "ent": "ratchet", "catw": "tool"}, {"id": 544, "ent": "rival", "catw": "weapon"}, {"id": 545, "ent": "candlestick", "catw": "device"}, {"id": 546, "ent": "helmsman", "catw": "boat"}, {"id": 547, "ent": "suitor", "catw": "person"}, {"id": 548, "ent": "greenhouse", "catw": "device"}, {"id": 549, "ent": "socialist", "catw": "person"}, {"id": 550, "ent": "prince", "catw": "person"}, {"id": 551, "ent": "pulley", "catw": "footwear"}, {"id": 552, "ent": "parachute", "catw": "device"}, {"id": 553, "ent": "crocodile", "catw": "body part"}, {"id": 554, "ent": "innocent", "catw": "man-made object"}, {"id": 555, "ent": "conch", "catw": "living thing"}, {"id": 556, "ent": "electrode", "catw": "car"}, {"id": 557, "ent": "raglan", "catw": "furniture"}, {"id": 558, "ent": "nacho", "catw": "material"}, {"id": 559, "ent": "surgeon", "catw": "worker"}, {"id": 560, "ent": "artist", "catw": "fish"}, {"id": 561, "ent": "claimant", "catw": "disease"}, {"id": 562, "ent": "ibis", "catw": "footwear"}, {"id": 563, "ent": "notary", "catw": "person"}, {"id": 564, "ent": "motorboat", "catw": "machine"}, {"id": 565, "ent": "loom", "catw": "man-made object"}, {"id": 566, "ent": "firearm", "catw": "weapon"}, {"id": 567, "ent": "haggis", "catw": "organ"}, {"id": 568, "ent": "brogan", "catw": "food"}, {"id": 569, "ent": "tam", "catw": "dish"}, {"id": 570, "ent": "motorcyclist", "catw": "sport"}, {"id": 571, "ent": "alumnus", "catw": "plant"}, {"id": 572, "ent": "dildo", "catw": "equipment"}, {"id": 573, "ent": "tights", "catw": "vegetable"}, {"id": 574, "ent": "doorbell", "catw": "boat"}, {"id": 575, "ent": "letterman", "catw": "reptile"}, {"id": 576, "ent": "athlete", "catw": "worker"}, {"id": 577, "ent": "hostage", "catw": "game"}, {"id": 578, "ent": "empty", "catw": "container"}, {"id": 579, "ent": "spacecraft", "catw": "machine"}, {"id": 580, "ent": "kilt", "catw": "boat"}, {"id": 581, "ent": "shook", "catw": "body part"}, {"id": 582, "ent": "headscarf", "catw": "clothing"}, {"id": 583, "ent": "geranium", "catw": "animal"}, {"id": 584, "ent": "mangrove", "catw": "weapon"}, {"id": 585, "ent": "librarian", "catw": "food"}, {"id": 586, "ent": "restaurateur", "catw": "athlete"}, {"id": 587, "ent": "perfectionist", "catw": "living thing"}, {"id": 588, "ent": "flashlight", "catw": "tool"}, {"id": 589, "ent": "groundhog", "catw": "man-made object"}, {"id": 590, "ent": "contemplative", "catw": "footwear"}, {"id": 591, "ent": "robber", "catw": "living thing"}, {"id": 592, "ent": "tequila", "catw": "plant"}, {"id": 593, "ent": "expert", "catw": "person"}, {"id": 594, "ent": "wheelchair", "catw": "game"}, {"id": 595, "ent": "cuttlefish", "catw": "artist"}, {"id": 596, "ent": "traveler", "catw": "game"}, {"id": 597, "ent": "thermos", "catw": "plant"}, {"id": 598, "ent": "customer", "catw": "weapon"}, {"id": 599, "ent": "loner", "catw": "living thing"}, {"id": 600, "ent": "slipknot", "catw": "material"}, {"id": 601, "ent": "landlady", "catw": "game"}, {"id": 602, "ent": "oppressor", "catw": "device"}, {"id": 603, "ent": "alderman", "catw": "boat"}, {"id": 604, "ent": "fife", "catw": "fabric"}, {"id": 605, "ent": "loudspeaker", "catw": "disease"}, {"id": 606, "ent": "revisionist", "catw": "artist"}, {"id": 607, "ent": "plenipotentiary", "catw": "mammal"}, {"id": 608, "ent": "houseboat", "catw": "vehicle"}, {"id": 609, "ent": "optimist", "catw": "body part"}, {"id": 610, "ent": "primrose", "catw": "flower"}, {"id": 611, "ent": "moray", "catw": "insect"}, {"id": 612, "ent": "roebuck", "catw": "animal"}, {"id": 613, "ent": "cafe", "catw": "man-made object"}, {"id": 614, "ent": "proletarian", "catw": "worker"}, {"id": 615, "ent": "limiter", "catw": "man-made object"}, {"id": 616, "ent": "brogan", "catw": "man-made object"}, {"id": 617, "ent": "wingman", "catw": "game"}, {"id": 618, "ent": "robber", "catw": "game"}, {"id": 619, "ent": "aquarium", "catw": "boat"}, {"id": 620, "ent": "handkerchief", "catw": "device"}, {"id": 621, "ent": "legislator", "catw": "worker"}, {"id": 622, "ent": "preteen", "catw": "person"}, {"id": 623, "ent": "individualist", "catw": "worker"}, {"id": 624, "ent": "detonator", "catw": "furniture"}, {"id": 625, "ent": "banjo", "catw": "fish"}, {"id": 626, "ent": "neurosurgeon", "catw": "footwear"}, {"id": 627, "ent": "sunflower", "catw": "living thing"}, {"id": 628, "ent": "kylie", "catw": "aircraft"}, {"id": 629, "ent": "nawab", "catw": "clothing"}, {"id": 630, "ent": "assistant", "catw": "person"}, {"id": 631, "ent": "cantilever", "catw": "body part"}, {"id": 632, "ent": "libertine", "catw": "fish"}, {"id": 633, "ent": "chronicler", "catw": "artist"}, {"id": 634, "ent": "earl", "catw": "boat"}, {"id": 635, "ent": "abbess", "catw": "building"}, {"id": 636, "ent": "murine", "catw": "food"}, {"id": 637, "ent": "stole", "catw": "weapon"}, {"id": 638, "ent": "stapler", "catw": "boat"}, {"id": 639, "ent": "ski", "catw": "device"}, {"id": 640, "ent": "materiel", "catw": "equipment"}, {"id": 641, "ent": "rheumatic", "catw": "living thing"}, {"id": 642, "ent": "margarita", "catw": "dish"}, {"id": 643, "ent": "dredge", "catw": "animal"}, {"id": 644, "ent": "private", "catw": "fruit"}, {"id": 645, "ent": "chalet", "catw": "car"}, {"id": 646, "ent": "locket", "catw": "device"}, {"id": 647, "ent": "earl", "catw": "animal"}, {"id": 648, "ent": "blazer", "catw": "weapon"}, {"id": 649, "ent": "jammer", "catw": "body part"}, {"id": 650, "ent": "crustacean", "catw": "food"}, {"id": 651, "ent": "android", "catw": "worker"}, {"id": 652, "ent": "cowgirl", "catw": "animal"}, {"id": 653, "ent": "entrepreneur", "catw": "food"}, {"id": 654, "ent": "implant", "catw": "body part"}, {"id": 655, "ent": "camcorder", "catw": "device"}, {"id": 656, "ent": "macadamia", "catw": "living thing"}, {"id": 657, "ent": "waterfowl", "catw": "game"}, {"id": 658, "ent": "celibate", "catw": "person"}, {"id": 659, "ent": "rhinoceros", "catw": "person"}, {"id": 660, "ent": "blazer", "catw": "man-made object"}, {"id": 661, "ent": "retriever", "catw": "game"}, {"id": 662, "ent": "antelope", "catw": "footwear"}, {"id": 663, "ent": "wally", "catw": "sport"}, {"id": 664, "ent": "usurper", "catw": "game"}, {"id": 665, "ent": "cultivar", "catw": "material"}, {"id": 666, "ent": "chalet", "catw": "game"}, {"id": 667, "ent": "kitchen", "catw": "equipment"}, {"id": 668, "ent": "matchbox", "catw": "toy"}, {"id": 669, "ent": "homeowner", "catw": "person"}, {"id": 670, "ent": "candlestick", "catw": "tool"}, {"id": 671, "ent": "priestess", "catw": "dog"}, {"id": 672, "ent": "toddler", "catw": "mammal"}, {"id": 673, "ent": "detonator", "catw": "man-made object"}, {"id": 674, "ent": "ax", "catw": "man-made object"}, {"id": 675, "ent": "lingerie", "catw": "equipment"}, {"id": 676, "ent": "biochemist", "catw": "worker"}, {"id": 677, "ent": "clary", "catw": "weapon"}, {"id": 678, "ent": "juvenile", "catw": "person"}, {"id": 679, "ent": "psychologist", "catw": "worker"}, {"id": 680, "ent": "obstetrician", "catw": "person"}, {"id": 681, "ent": "biologist", "catw": "worker"}, {"id": 682, "ent": "camcorder", "catw": "room"}, {"id": 683, "ent": "aristocrat", "catw": "fruit"}, {"id": 684, "ent": "tripod", "catw": "equipment"}, {"id": 685, "ent": "ashtray", "catw": "metal"}, {"id": 686, "ent": "atheist", "catw": "dog"}, {"id": 687, "ent": "airship", "catw": "body part"}, {"id": 688, "ent": "greenhouse", "catw": "container"}, {"id": 689, "ent": "canoe", "catw": "vehicle"}, {"id": 690, "ent": "ship", "catw": "tool"}, {"id": 691, "ent": "jodhpur", "catw": "vehicle"}, {"id": 692, "ent": "coffin", "catw": "footwear"}, {"id": 693, "ent": "wrestler", "catw": "athlete"}, {"id": 694, "ent": "dominatrix", "catw": "vehicle"}, {"id": 695, "ent": "caribou", "catw": "game"}, {"id": 696, "ent": "harmonica", "catw": "equipment"}, {"id": 697, "ent": "genitalia", "catw": "vegetable"}, {"id": 698, "ent": "anesthesiologist", "catw": "worker"}, {"id": 699, "ent": "golf", "catw": "equipment"}]''')
NEW = json.loads(r'''[{"id": 1000, "ent": "racist", "catw": "plant"}, {"id": 1001, "ent": "goggles", "catw": "equipment"}, {"id": 1002, "ent": "swatch", "catw": "man-made object"}, {"id": 1003, "ent": "payer", "catw": "machine"}, {"id": 1004, "ent": "mater", "catw": "material"}, {"id": 1005, "ent": "fancier", "catw": "game"}, {"id": 1006, "ent": "electrician", "catw": "plant"}, {"id": 1007, "ent": "crucible", "catw": "man-made object"}, {"id": 1008, "ent": "consul", "catw": "tool"}, {"id": 1009, "ent": "dropper", "catw": "tool"}, {"id": 1010, "ent": "juror", "catw": "material"}, {"id": 1011, "ent": "abbot", "catw": "container"}, {"id": 1012, "ent": "viper", "catw": "weapon"}, {"id": 1013, "ent": "midge", "catw": "living thing"}, {"id": 1014, "ent": "godmother", "catw": "bird"}, {"id": 1015, "ent": "locator", "catw": "metal"}, {"id": 1016, "ent": "bassist", "catw": "living thing"}, {"id": 1017, "ent": "immune", "catw": "sport"}, {"id": 1018, "ent": "fink", "catw": "material"}, {"id": 1019, "ent": "cosmopolitan", "catw": "drink"}, {"id": 1020, "ent": "liza", "catw": "person"}, {"id": 1021, "ent": "stallion", "catw": "mammal"}, {"id": 1022, "ent": "vassal", "catw": "dog"}, {"id": 1023, "ent": "dragoon", "catw": "living thing"}, {"id": 1024, "ent": "traditionalist", "catw": "body part"}, {"id": 1025, "ent": "stallion", "catw": "living thing"}, {"id": 1026, "ent": "headmaster", "catw": "container"}, {"id": 1027, "ent": "tennis", "catw": "container"}, {"id": 1028, "ent": "godmother", "catw": "material"}, {"id": 1029, "ent": "magdalen", "catw": "boat"}, {"id": 1030, "ent": "pervert", "catw": "device"}, {"id": 1031, "ent": "neoliberal", "catw": "tool"}, {"id": 1032, "ent": "villager", "catw": "living thing"}, {"id": 1033, "ent": "salad", "catw": "machine"}, {"id": 1034, "ent": "supremacist", "catw": "person"}, {"id": 1035, "ent": "thermostat", "catw": "metal"}, {"id": 1036, "ent": "guan", "catw": "toy"}, {"id": 1037, "ent": "bodice", "catw": "clothing"}, {"id": 1038, "ent": "clarinet", "catw": "living thing"}, {"id": 1039, "ent": "tapeworm", "catw": "vegetable"}, {"id": 1040, "ent": "scold", "catw": "person"}, {"id": 1041, "ent": "ratatouille", "catw": "insect"}, {"id": 1042, "ent": "panelist", "catw": "metal"}, {"id": 1043, "ent": "vegan", "catw": "machine"}, {"id": 1044, "ent": "fern", "catw": "metal"}, {"id": 1045, "ent": "heiress", "catw": "boat"}, {"id": 1046, "ent": "mosque", "catw": "container"}, {"id": 1047, "ent": "literate", "catw": "vegetable"}, {"id": 1048, "ent": "machinist", "catw": "person"}, {"id": 1049, "ent": "sickle", "catw": "disease"}, {"id": 1050, "ent": "heiress", "catw": "fruit"}, {"id": 1051, "ent": "religious", "catw": "material"}, {"id": 1052, "ent": "spelt", "catw": "food"}, {"id": 1053, "ent": "paella", "catw": "body part"}, {"id": 1054, "ent": "laser", "catw": "game"}, {"id": 1055, "ent": "tamer", "catw": "dish"}, {"id": 1056, "ent": "executor", "catw": "worker"}, {"id": 1057, "ent": "islander", "catw": "mammal"}, {"id": 1058, "ent": "urchin", "catw": "worker"}, {"id": 1059, "ent": "obsessive", "catw": "car"}, {"id": 1060, "ent": "civilian", "catw": "tree"}, {"id": 1061, "ent": "sword", "catw": "man-made object"}, {"id": 1062, "ent": "slingshot", "catw": "musical instrument"}, {"id": 1063, "ent": "cappuccino", "catw": "living thing"}, {"id": 1064, "ent": "investor", "catw": "mammal"}, {"id": 1065, "ent": "betrothed", "catw": "game"}, {"id": 1066, "ent": "condor", "catw": "animal"}, {"id": 1067, "ent": "magistrate", "catw": "living thing"}, {"id": 1068, "ent": "turtleneck", "catw": "plant"}, {"id": 1069, "ent": "mastodon", "catw": "tool"}, {"id": 1070, "ent": "luge", "catw": "sport"}, {"id": 1071, "ent": "urologist", "catw": "person"}, {"id": 1072, "ent": "tray", "catw": "dish"}, {"id": 1073, "ent": "landlord", "catw": "game"}, {"id": 1074, "ent": "whisperer", "catw": "body part"}, {"id": 1075, "ent": "paella", "catw": "dish"}, {"id": 1076, "ent": "sandal", "catw": "room"}, {"id": 1077, "ent": "kicker", "catw": "furniture"}, {"id": 1078, "ent": "metronome", "catw": "device"}, {"id": 1079, "ent": "portable", "catw": "man-made object"}, {"id": 1080, "ent": "badminton", "catw": "container"}, {"id": 1081, "ent": "pauper", "catw": "person"}, {"id": 1082, "ent": "bigot", "catw": "game"}, {"id": 1083, "ent": "taxpayer", "catw": "vegetable"}, {"id": 1084, "ent": "anthropologist", "catw": "man-made object"}, {"id": 1085, "ent": "farmhouse", "catw": "fruit"}, {"id": 1086, "ent": "stockman", "catw": "tree"}, {"id": 1087, "ent": "tambourine", "catw": "man-made object"}, {"id": 1088, "ent": "outcast", "catw": "person"}, {"id": 1089, "ent": "contemporary", "catw": "game"}, {"id": 1090, "ent": "caliper", "catw": "reptile"}, {"id": 1091, "ent": "sprinkler", "catw": "tool"}, {"id": 1092, "ent": "screenwriter", "catw": "person"}, {"id": 1093, "ent": "lily", "catw": "reptile"}, {"id": 1094, "ent": "kinsman", "catw": "game"}, {"id": 1095, "ent": "elitist", "catw": "game"}, {"id": 1096, "ent": "archivist", "catw": "person"}, {"id": 1097, "ent": "tadpole", "catw": "building"}, {"id": 1098, "ent": "magneto", "catw": "sport"}, {"id": 1099, "ent": "catamaran", "catw": "vehicle"}, {"id": 1100, "ent": "carburetor", "catw": "animal"}, {"id": 1101, "ent": "illiterate", "catw": "living thing"}, {"id": 1102, "ent": "transformer", "catw": "sport"}, {"id": 1103, "ent": "alarmist", "catw": "person"}, {"id": 1104, "ent": "swatch", "catw": "living thing"}, {"id": 1105, "ent": "convalescent", "catw": "flower"}, {"id": 1106, "ent": "guildhall", "catw": "room"}, {"id": 1107, "ent": "sprinter", "catw": "tool"}, {"id": 1108, "ent": "pug", "catw": "man-made object"}, {"id": 1109, "ent": "practitioner", "catw": "person"}, {"id": 1110, "ent": "sheriff", "catw": "worker"}, {"id": 1111, "ent": "skier", "catw": "sport"}, {"id": 1112, "ent": "resistor", "catw": "device"}, {"id": 1113, "ent": "netball", "catw": "sport"}, {"id": 1114, "ent": "emeritus", "catw": "person"}, {"id": 1115, "ent": "spitz", "catw": "boat"}, {"id": 1116, "ent": "nettle", "catw": "equipment"}, {"id": 1117, "ent": "artiste", "catw": "body part"}, {"id": 1118, "ent": "laptop", "catw": "equipment"}, {"id": 1119, "ent": "cymbal", "catw": "animal"}, {"id": 1120, "ent": "cricketer", "catw": "person"}, {"id": 1121, "ent": "hotel", "catw": "room"}, {"id": 1122, "ent": "compatriot", "catw": "boat"}, {"id": 1123, "ent": "thistle", "catw": "flower"}, {"id": 1124, "ent": "colleen", "catw": "clothing"}, {"id": 1125, "ent": "risotto", "catw": "dish"}, {"id": 1126, "ent": "pimp", "catw": "person"}, {"id": 1127, "ent": "mantis", "catw": "animal"}, {"id": 1128, "ent": "giraffe", "catw": "fish"}, {"id": 1129, "ent": "armature", "catw": "man-made object"}, {"id": 1130, "ent": "mollie", "catw": "dog"}, {"id": 1131, "ent": "azalea", "catw": "fish"}, {"id": 1132, "ent": "condom", "catw": "device"}, {"id": 1133, "ent": "mastiff", "catw": "game"}, {"id": 1134, "ent": "butch", "catw": "artist"}, {"id": 1135, "ent": "prefect", "catw": "living thing"}, {"id": 1136, "ent": "breadwinner", "catw": "vegetable"}, {"id": 1137, "ent": "gnu", "catw": "dog"}, {"id": 1138, "ent": "supercharger", "catw": "man-made object"}, {"id": 1139, "ent": "blaster", "catw": "living thing"}, {"id": 1140, "ent": "serviceman", "catw": "material"}, {"id": 1141, "ent": "depressive", "catw": "disease"}, {"id": 1142, "ent": "collie", "catw": "animal"}, {"id": 1143, "ent": "hosiery", "catw": "man-made object"}, {"id": 1144, "ent": "buggy", "catw": "container"}, {"id": 1145, "ent": "tetra", "catw": "mammal"}, {"id": 1146, "ent": "pullover", "catw": "game"}, {"id": 1147, "ent": "yucca", "catw": "weapon"}, {"id": 1148, "ent": "asthmatic", "catw": "disease"}, {"id": 1149, "ent": "stockman", "catw": "fabric"}, {"id": 1150, "ent": "scientist", "catw": "person"}, {"id": 1151, "ent": "acacia", "catw": "tree"}, {"id": 1152, "ent": "godmother", "catw": "boat"}, {"id": 1153, "ent": "cafeteria", "catw": "building"}, {"id": 1154, "ent": "eclectic", "catw": "furniture"}, {"id": 1155, "ent": "hibiscus", "catw": "flower"}, {"id": 1156, "ent": "assemblyman", "catw": "living thing"}, {"id": 1157, "ent": "everyman", "catw": "tree"}, {"id": 1158, "ent": "sniffer", "catw": "vegetable"}, {"id": 1159, "ent": "fiance", "catw": "game"}, {"id": 1160, "ent": "caddy", "catw": "container"}, {"id": 1161, "ent": "swami", "catw": "vehicle"}, {"id": 1162, "ent": "reptile", "catw": "animal"}, {"id": 1163, "ent": "hourglass", "catw": "food"}, {"id": 1164, "ent": "baroness", "catw": "sport"}, {"id": 1165, "ent": "boyfriend", "catw": "musical instrument"}, {"id": 1166, "ent": "omelet", "catw": "weapon"}, {"id": 1167, "ent": "ant", "catw": "reptile"}, {"id": 1168, "ent": "breeder", "catw": "room"}, {"id": 1169, "ent": "lidar", "catw": "footwear"}, {"id": 1170, "ent": "penguin", "catw": "plant"}, {"id": 1171, "ent": "batik", "catw": "metal"}, {"id": 1172, "ent": "pauper", "catw": "living thing"}, {"id": 1173, "ent": "mahatma", "catw": "bird"}, {"id": 1174, "ent": "diocesan", "catw": "person"}, {"id": 1175, "ent": "shrub", "catw": "game"}, {"id": 1176, "ent": "padlock", "catw": "tool"}, {"id": 1177, "ent": "astrologer", "catw": "animal"}, {"id": 1178, "ent": "harpsichord", "catw": "equipment"}, {"id": 1179, "ent": "mediator", "catw": "living thing"}, {"id": 1180, "ent": "fanatic", "catw": "material"}, {"id": 1181, "ent": "tamer", "catw": "sport"}, {"id": 1182, "ent": "weevil", "catw": "person"}, {"id": 1183, "ent": "panelist", "catw": "living thing"}, {"id": 1184, "ent": "hacksaw", "catw": "vegetable"}, {"id": 1185, "ent": "applicant", "catw": "living thing"}, {"id": 1186, "ent": "reactionary", "catw": "boat"}, {"id": 1187, "ent": "tern", "catw": "animal"}, {"id": 1188, "ent": "registrant", "catw": "man-made object"}, {"id": 1189, "ent": "marigold", "catw": "person"}, {"id": 1190, "ent": "barometer", "catw": "device"}, {"id": 1191, "ent": "paratrooper", "catw": "insect"}, {"id": 1192, "ent": "chump", "catw": "game"}, {"id": 1193, "ent": "vertebrate", "catw": "mammal"}, {"id": 1194, "ent": "slingshot", "catw": "fish"}, {"id": 1195, "ent": "walrus", "catw": "fruit"}, {"id": 1196, "ent": "workaholic", "catw": "worker"}, {"id": 1197, "ent": "guesthouse", "catw": "metal"}, {"id": 1198, "ent": "housemate", "catw": "musical instrument"}, {"id": 1199, "ent": "groupie", "catw": "person"}, {"id": 1200, "ent": "abalone", "catw": "food"}, {"id": 1201, "ent": "seller", "catw": "artist"}, {"id": 1202, "ent": "grenade", "catw": "man-made object"}, {"id": 1203, "ent": "zebra", "catw": "vegetable"}, {"id": 1204, "ent": "serviceman", "catw": "tree"}, {"id": 1205, "ent": "colonel", "catw": "living thing"}, {"id": 1206, "ent": "colonial", "catw": "metal"}, {"id": 1207, "ent": "brew", "catw": "disease"}, {"id": 1208, "ent": "guan", "catw": "bird"}, {"id": 1209, "ent": "skiff", "catw": "container"}, {"id": 1210, "ent": "gander", "catw": "living thing"}, {"id": 1211, "ent": "cockatoo", "catw": "reptile"}, {"id": 1212, "ent": "shaman", "catw": "living thing"}, {"id": 1213, "ent": "businesswoman", "catw": "worker"}, {"id": 1214, "ent": "nomad", "catw": "worker"}, {"id": 1215, "ent": "armature", "catw": "equipment"}, {"id": 1216, "ent": "turbine", "catw": "man-made object"}, {"id": 1217, "ent": "scythe", "catw": "equipment"}, {"id": 1218, "ent": "ale", "catw": "drink"}, {"id": 1219, "ent": "cappuccino", "catw": "person"}, {"id": 1220, "ent": "vine", "catw": "disease"}, {"id": 1221, "ent": "bookshelf", "catw": "furniture"}, {"id": 1222, "ent": "dirk", "catw": "worker"}, {"id": 1223, "ent": "essayist", "catw": "material"}, {"id": 1224, "ent": "clarion", "catw": "metal"}, {"id": 1225, "ent": "boudoir", "catw": "game"}, {"id": 1226, "ent": "adjuster", "catw": "flower"}, {"id": 1227, "ent": "airbrush", "catw": "building"}, {"id": 1228, "ent": "schmuck", "catw": "material"}, {"id": 1229, "ent": "mystic", "catw": "room"}, {"id": 1230, "ent": "transponder", "catw": "equipment"}, {"id": 1231, "ent": "purist", "catw": "drink"}, {"id": 1232, "ent": "rani", "catw": "mammal"}, {"id": 1233, "ent": "overseer", "catw": "living thing"}, {"id": 1234, "ent": "locator", "catw": "tool"}, {"id": 1235, "ent": "pinto", "catw": "flower"}, {"id": 1236, "ent": "reeve", "catw": "bird"}, {"id": 1237, "ent": "valet", "catw": "device"}, {"id": 1238, "ent": "evergreen", "catw": "plant"}, {"id": 1239, "ent": "fisherman", "catw": "game"}, {"id": 1240, "ent": "chipmunk", "catw": "mammal"}, {"id": 1241, "ent": "magdalen", "catw": "person"}, {"id": 1242, "ent": "sahib", "catw": "vegetable"}, {"id": 1243, "ent": "marauder", "catw": "boat"}, {"id": 1244, "ent": "duce", "catw": "living thing"}, {"id": 1245, "ent": "bobby", "catw": "animal"}, {"id": 1246, "ent": "scapegoat", "catw": "bird"}, {"id": 1247, "ent": "sinner", "catw": "game"}, {"id": 1248, "ent": "mastodon", "catw": "flower"}, {"id": 1249, "ent": "breeder", "catw": "person"}, {"id": 1250, "ent": "bathtub", "catw": "equipment"}, {"id": 1251, "ent": "infielder", "catw": "food"}, {"id": 1252, "ent": "mustang", "catw": "living thing"}, {"id": 1253, "ent": "surfer", "catw": "athlete"}, {"id": 1254, "ent": "converter", "catw": "equipment"}, {"id": 1255, "ent": "prior", "catw": "living thing"}, {"id": 1256, "ent": "prowler", "catw": "animal"}, {"id": 1257, "ent": "juror", "catw": "worker"}, {"id": 1258, "ent": "dermatologist", "catw": "material"}, {"id": 1259, "ent": "policeman", "catw": "food"}, {"id": 1260, "ent": "triceps", "catw": "weapon"}, {"id": 1261, "ent": "tun", "catw": "boat"}, {"id": 1262, "ent": "speedometer", "catw": "equipment"}, {"id": 1263, "ent": "beggar", "catw": "sport"}, {"id": 1264, "ent": "workaholic", "catw": "bird"}, {"id": 1265, "ent": "lathe", "catw": "device"}, {"id": 1266, "ent": "gland", "catw": "organ"}, {"id": 1267, "ent": "starling", "catw": "vegetable"}, {"id": 1268, "ent": "ladle", "catw": "tool"}, {"id": 1269, "ent": "pontifical", "catw": "insect"}, {"id": 1270, "ent": "collie", "catw": "vegetable"}, {"id": 1271, "ent": "dramatist", "catw": "equipment"}, {"id": 1272, "ent": "goldfish", "catw": "animal"}, {"id": 1273, "ent": "corporal", "catw": "weapon"}, {"id": 1274, "ent": "overseer", "catw": "worker"}, {"id": 1275, "ent": "shill", "catw": "artist"}, {"id": 1276, "ent": "prisoner", "catw": "game"}, {"id": 1277, "ent": "contemporary", "catw": "artist"}, {"id": 1278, "ent": "gorilla", "catw": "athlete"}, {"id": 1279, "ent": "cursor", "catw": "tool"}, {"id": 1280, "ent": "socialite", "catw": "living thing"}, {"id": 1281, "ent": "milo", "catw": "living thing"}, {"id": 1282, "ent": "techie", "catw": "person"}, {"id": 1283, "ent": "madder", "catw": "living thing"}, {"id": 1284, "ent": "troublemaker", "catw": "vegetable"}, {"id": 1285, "ent": "bricklayer", "catw": "athlete"}, {"id": 1286, "ent": "guitar", "catw": "musical instrument"}, {"id": 1287, "ent": "fizz", "catw": "game"}, {"id": 1288, "ent": "pantry", "catw": "room"}, {"id": 1289, "ent": "bisexual", "catw": "living thing"}, {"id": 1290, "ent": "jaguar", "catw": "animal"}, {"id": 1291, "ent": "layman", "catw": "plant"}, {"id": 1292, "ent": "namesake", "catw": "sport"}, {"id": 1293, "ent": "moped", "catw": "container"}, {"id": 1294, "ent": "megaphone", "catw": "food"}, {"id": 1295, "ent": "skeptic", "catw": "boat"}, {"id": 1296, "ent": "novelist", "catw": "artist"}, {"id": 1297, "ent": "condom", "catw": "equipment"}, {"id": 1298, "ent": "yacht", "catw": "dog"}, {"id": 1299, "ent": "vine", "catw": "plant"}]''')
ITEMS = OLD + NEW
L = json.load(open(E8N))['L']
print(len(OLD), '+', len(NEW), '問  語の意味の一覧', len(L))

## 1. WordNet の名詞と SemCor の回数（GPUは使わない、1〜3分）

In [ ]:
import nltk
nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
from nltk.corpus import wordnet as wn
WN = semcor_items(wn, n=2000, seed=0)
print("①の語:", len(WN), " 例:", WN[0]["w"], [d[:40] for d in WN[0]["defs"]][:3])

## 2. 語と説明文の近さ（S1、GPU、1〜2分）

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer(EMB_ID, device="cuda")
S1 = []
for it in WN:
    v = emb.encode([it["w"]] + it["defs"], normalize_embeddings=True, show_progress_bar=False)
    S1.append((v[1:] @ v[0]).round(4).tolist())
del emb; torch.cuda.empty_cache()
print("S1 済み", len(S1))

## 3. Gemma に意味を選ばせる（①・②・参考R4、GPU、15〜30分）

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
NUM = [tok.encode(str(i), add_special_tokens=False)[0] for i in range(1, MAX_SENSES + 1)]
assert len(set(NUM)) == MAX_SENSES

def ask(prompts, sizes, ck, B=16):
    if os.path.exists(ck):
        c = json.load(open(ck))
        if c.get("n") == len(prompts) and c.get("head") == prompts[0][:200]:
            print("ドライブから読み込み", ck); return c["out"]
    out = []; t = time.time()
    for i in range(0, len(prompts), B):
        ps = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in prompts[i:i + B]]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
        for row, n in zip(last, sizes[i:i + B]):
            out.append(torch.softmax(row[NUM[:n]], -1).cpu().numpy().round(4).tolist())
        if (i // B) % 100 == 0: print(f"{i}/{len(prompts)}  {time.time() - t:.0f}秒")
    json.dump({"n": len(prompts), "head": prompts[0][:200], "out": out}, open(ck, "w"))
    return out

# ① WordNet：くじの順とその逆順の2通り
J1 = jobs_usual([(it["w"], it["defs"]) for it in WN], seed=0)
P1 = ask([j[0] for j in J1], [len(j[1]) for j in J1], f"{OUT}/ck_wn.json")
S2 = merge(J1, P1, len(WN))
# ② E8n の物の語（意味が2つ以上の物だけ）
ENTS = sorted({it["ent"] for it in ITEMS if len(L.get(it["ent"], [])) >= 2})
J2 = jobs_usual([(w, [x["def"] for x in L[w]]) for w in ENTS], seed=0)
P2 = ask([j[0] for j in J2], [len(j[1]) for j in J2], f"{OUT}/ck_ent.json")
R2 = dict(zip(ENTS, merge(J2, P2, len(ENTS))))
# 参考 R4：分類を見せて選ばせる（主張には使わない）
IT4 = [it for it in ITEMS if len(L.get(it["ent"], [])) >= 2]
J4 = jobs_leak([(it["ent"], [x["def"] for x in L[it["ent"]]], it["catw"]) for it in IT4], seed=0)
P4 = ask([j[0] for j in J4], [len(j[1]) for j in J4], f"{OUT}/ck_leak.json")
R4 = {it["id"]: p for it, p in zip(IT4, merge(J4, P4, len(IT4)))}
del gm; torch.cuda.empty_cache()
print("S2 一番上に並べた候補を選んだ割合（くじの順の1通り目、並べ順の癖の目安）:", np.mean([int(np.argmax(p)) == 0 for p in P1[0::2]]).round(3))
print("R2 Wiktionaryの最初の意味を選んだ割合:", np.mean([int(np.argmax(v)) == 0 for v in R2.values()]).round(3))

## 4. 保存

In [ ]:
res = {"meta": {"gemma": GEMMA_ID, "emb": EMB_ID, "e8n": "E8n_results/E8n_results.json", "date": time.strftime("%Y-%m-%d"),
                "wordnet": "nltk wordnet 3.0, lemma.count()", "n_wn": len(WN), "n_ent": len(ENTS)},
       "wn": [dict(it, s1=s1, s2=s2) for it, s1, s2 in zip(WN, S1, S2)],
       "ent_p": R2, "leak_p": {str(k): v for k, v in R4.items()}}
path = f"{OUT}/E11-1_results.json"
json.dump(res, open(path, "w"), ensure_ascii=False)
print("保存:", path, round(os.path.getsize(path) / 1e6, 2), "MB")